# AFM Laser Position Sweep Automation — v4

Automated workflow for Asylum Research AFM systems:
1. Move laser to position
2. Capture optical image
3. AutoWedge + InvOLS calibration
4. Engage and tune eigenmode (~400 kHz)
5. Acquire AFM image at 35 kHz
6. Restore drive frequency to 400 kHz
7. Repeat for all positions

**Single load, single DC bias** (no sweep over those parameters).

## 1. Imports and Setup

In [38]:
import numpy as np
import time
import os
import re
from igor2 import binarywave
import matplotlib.pyplot as plt
from PIL import Image

# For Igor Pro COM connection (Windows only)
import win32com.client

## 2. Connect to Igor Pro

In [39]:
# Connect to Igor Pro via COM
igor = win32com.client.Dispatch("IgorPro.Application")
print("Connected to Igor Pro")

Connected to Igor Pro


## 3. Define Experiment Parameters

**Modify these parameters for your experiment:**

In [40]:
# =============================================================================
# FILE PATHS
# =============================================================================
file_loc = r'D:\User Data\MARTI\2026\April\16th\Adama vs Load'  # Working directory
base_filename = 'LaserSweep'                 # Base name for saved files
log_filename = r'D:\User Data\MARTI\2026\April\16th\Adama vs Load'  # Log file (no extension)

# =============================================================================
# LASER SWEEP PARAMETERS
# =============================================================================
step_um = 0.5 # Step size in micrometers
num_steps = 100       # Number of positions to measure

# =============================================================================
# EIGENMODE FREQUENCY (Hz)
# =============================================================================
# Set this manually in Igor before running - this is just for reference/logging
eigenmode_center = 400000    # Contact resonance ~355 kHz

# =============================================================================
# ENGAGEMENT PARAMETERS
# =============================================================================
load_setpoint = 0.5     # Deflection setpoint in Volts

# =============================================================================
# AUTOWEDGE OPTION
# =============================================================================
use_autowedge = True  # Set to True to enable AutoWedge after InvOLS
autowedge_pause = 15.0  # Seconds to wait after AutoWedge

# =============================================================================
# TUNING PARAMETERS
# =============================================================================
tune_settling_time = 2.0     # Seconds to wait after tune

# =============================================================================
# INVOLS VALIDATION RANGE
# =============================================================================
invols_min = 4e-8   # Minimum valid InvOLS (m/V)
invols_max = 10e-7  # Maximum valid InvOLS (m/V)

In [41]:
# Print configuration summary
print("=" * 60)
print("EXPERIMENT CONFIGURATION")
print("=" * 60)
print(f"\nFile Location: {file_loc}")
print(f"Base Filename: {base_filename}")
print(f"\nLaser Sweep:")
print(f"  Step size: {step_um} µm")
print(f"  Number of steps: {num_steps}")
print(f"  Total range: {step_um * (num_steps - 1)} µm")
print(f"\nEigenmode Frequency (reference):")
print(f"  Center: {eigenmode_center/1000:.1f} kHz")
print(f"\nEngagement:")
print(f"  Load setpoint: {load_setpoint} V")
print(f"\nAutoWedge: {'ENABLED' if use_autowedge else 'DISABLED'}")
if use_autowedge:
    print(f"  Pause after: {autowedge_pause} s")

EXPERIMENT CONFIGURATION

File Location: D:\User Data\MARTI\2026\April\16th\Adama vs Load
Base Filename: LaserSweep

Laser Sweep:
  Step size: 0.5 µm
  Number of steps: 100
  Total range: 49.5 µm

Eigenmode Frequency (reference):
  Center: 400.0 kHz

Engagement:
  Load setpoint: 0.5 V

AutoWedge: ENABLED
  Pause after: 15.0 s


## 4. AFM Laser Sweep Automation Class

In [42]:
class AFMLaserSweepAutomation:
    """
    Automated laser position sweep with InvOLS calibration and eigenmode tuning.
    
    Workflow per position:
    1. Capture optical image
    2. Do force curve (InvOLS)
    3. Optional AutoWedge with pause
    4. Engage and tune eigenmode
    5. Move to next position
    """
    
    def __init__(self, igor, file_loc, base_filename, log_filename):
        """
        Initialize the automation class.
        """
        self.igor = igor
        self.file_loc = file_loc
        self.base_filename = base_filename
        self.log_filename = log_filename
        
        # Default parameters
        self.load_force_setpoint = 0.5
        self.tune_settling_time = 2.0
        self.invols_bounds = (4e-8, 10e-7)
        
        # AutoWedge settings
        self.use_autowedge = False
        self.autowedge_pause = 15.0
        
        # Eigenmode frequency (for reference/logging)
        self.eigenmode_center_freq = 70000
        
        # Results storage
        self.results = []
        
    def saveprint(self, text, f=None, verbose=True):
        """Print and optionally save to log file."""
        if verbose:
            print(text)
        if f:
            f.write(text + '\n')
            f.flush()
    
    def ex(self, variable="", panel="", val=0, string="", verbose=False):
        """
        Execute an Asylum Research control.
        Wrapper for ARExecuteControl Igor function.
        """
        execution_line = ""
        if verbose:
            execution_line += "print "
        execution_line += f'ARExecuteControl("{variable}", "{panel}", {val}, "{string}")'
        self.igor.Execute(execution_line)
        return execution_line
    
    def get_gmv(self):
        """Get all Master Variables from Igor."""
        data = self.igor.DataFolder(r"root:packages:MFP3D:Main:Variables").Wave("MasterVariablesWave")
        dim = data.GetDimensions()[1]
        gmv = {}
        for i in range(dim):
            gmv[data.DimensionLabel(0, i, 0)] = data.GetNumericWavePointValue(i)
        return gmv
    
    def set_folder(self):
        """Set the working folder in Igor."""
        file_loc = self.file_loc.split('\\', 1)
        file_loc_igor = file_loc[0] + file_loc[1].replace('\\', ':')
        if file_loc_igor[-1] != ':':
            file_loc_igor += ':'
        
        self.igor.Execute(f'root:Packages:MFP3D:Main:Strings:GlobalStrings[20] = "{file_loc_igor}"')
        self.igor.Execute(f'InsertNewPathInHistory("{file_loc_igor}")')
        self.igor.Execute(f'root:Packages:MFP3D:Main:Strings:GlobalStrings[18] = "{file_loc_igor}"')
        self.igor.Execute(f'NewPath/O SaveImage "{file_loc_igor}"')
        self.igor.Execute(f'NewPath/O SaveForce "{file_loc_igor}"')
    
    def do_ld_move(self, microns_x, microns_y):
        """
        Move the laser position by specified microns.
        """
        self.igor.Execute(f'DoLDMove({microns_x}, {microns_y})')
        time.sleep(0.5)
    
    def move_laser_to_position(self, x_um, y_um, relative=True):
        """
        Move laser to specified position.
        """
        if relative:
            self.do_ld_move(x_um, y_um)
        else:
            self.igor.Execute("save_laser_position_in_igor()")
            current_x = self.igor.DataFolder(r"root").Wave("LDX_pos").GetNumericWavePointValue(0)
            current_y = self.igor.DataFolder(r"root").Wave("LDX_pos").GetNumericWavePointValue(1)
            dx = (x_um * 10 - current_x) / 10
            dy = (y_um * 10 - current_y) / 10
            self.do_ld_move(dx, dy)
    
    def _get_next_filename(self, base_name, extension):
        """Get the next available filename with incrementing suffix."""
        pattern = re.compile(rf"^{re.escape(base_name)}(\d{{4}})\..+$")
        max_index = -1
        
        for filename in os.listdir(self.file_loc):
            match = pattern.match(filename)
            if match:
                index = int(match.group(1))
                if index > max_index:
                    max_index = index
        
        next_index = max_index + 1
        full_path = os.path.join(self.file_loc, f"{base_name}{next_index:04d}{extension}")
        return full_path
    
    def capture_optical_image(self, position_label=""):
        """
        Capture an optical image from the video panel.
        """
        # Configure save location
        self.set_folder()
        
        # Capture the image using ARVideoButtonFunc
        self.igor.Execute('ARVideoButtonFunc("ARVCapture")')
        time.sleep(1)
        
        print(f"  Captured optical image")
        return "captured"
    
    def simple_engage(self, wait_time=5):
        """
        Engage using SimpleEngageMe function.
        """
        print("  Engaging with SimpleEngageMe...")
        self.igor.Execute('SimpleEngageMe("")')
        time.sleep(wait_time)
        print("  Engagement command sent")
        return True
   
    def withdraw(self):
        """Withdraw the tip from the surface."""
        print("  Withdrawing tip...")
        self.igor.Execute('DoScanFunc("StopEngageButton")')
        time.sleep(2)
        
    def measure_invols(self):
        """
        Measure and set InvOLS using a force curve.
        
        Returns
        -------
        float
            Measured InvOLS value in m/V
        """
        self.set_folder()
        os.chdir(self.file_loc)
        
        gmv = self.get_gmv()
        
        # Set trigger for force curve
        self.ex("TriggerChannelPopup_1", "MasterPanel", 0, "DeflVolts")
        self.ex("TriggerPointSetVar_1", "MasterPanel", gmv["DeflectionSetpointVolts"])
        
        # Set filename
        note = 'F'
        self.igor.Execute(f'root:Packages:MFP3D:Main:Variables:BaseName = "{note}{self.base_filename}"')
        self.igor.Execute('PV("BaseSuffix", 0000)')
        self.igor.Execute('ARCheckSuffix()')
        
        fname = self._get_next_filename(f'{note}{self.base_filename}', '.ibw')
        
        print(f"  Performing force curve: {os.path.basename(fname)}")
        
        # Execute single force curve
        self.ex("SingleForce_1", "MasterPanel", 1)
        time.sleep(8)
        
        # Load and analyze the force curve
        try:
            file = binarywave.load(fname)
            z = file['wave']['wData'][:, 4]
            defl = file['wave']['wData'][:, 1]
            
            max_i = np.argmax(defl)
            engage_i = np.argmin(defl[:max_i])
            
            # Get current InvOLS from file note
            curr_invols = float(str(file['wave']['note']).split('rInvOLS: ')[1].split('\\')[0])
            deflV = defl / curr_invols
            
            # Calculate new InvOLS from slope
            diff_i = max_i - engage_i
            i_1 = int(engage_i + 0.25 * diff_i)
            i_2 = int(max_i - 0.25 * diff_i)
            
            invols = (z[i_2] - z[i_1]) / (deflV[i_2] - deflV[i_1])
            
            print(f"  Measured InvOLS: {invols:.3e} m/V")
            
            # Set InvOLS if within bounds
            if self.invols_bounds[0] < invols < self.invols_bounds[1]:
                self.ex('InvOLSSetVar_1', 'MasterPanel', invols)
                print("  InvOLS set successfully")
            else:
                print(f"  WARNING: InvOLS {invols:.3e} outside valid range {self.invols_bounds}")
            
            return invols
            
        except Exception as e:
            print(f"  ERROR loading force curve: {e}")
            return None
    
    def do_autowedge(self):
        """
        Perform AutoWedge calibration with pause.
        """
        print(f"  Running AutoWedge...")
        self.igor.Execute("AutoWedge()")
        #self.igor.Execute("ZeroPD(\"\")")
        print(f"  Waiting {15}s for AutoWedge to settle...")
        time.sleep(15)
        print("  AutoWedge complete")
    
    def windows2igor(self, path):
        """Convert Windows path to Igor path format."""
        parts = path.split('\\', 1)
        if len(parts) == 2:
            igor_path = parts[0] + parts[1].replace('\\', ':')
        else:
            igor_path = path.replace('\\', ':')
        if igor_path[-1] != ':':
            igor_path += ':'
        return igor_path
    
    def do_tune(self, wait_time=5):
        """Perform a cantilever tune using CanttuneFunc."""
        self.igor.Execute('CanttuneFunc("DoTuneOnceButton")')
        time.sleep(wait_time)
    
    def save_tune(self, position_label="", scan_index=0):
        """
        Save the current tune data (Frequency, Phase, Amplitude) to a text file.
        """
        
        tune_filename = f"Tune_{self.base_filename}_{position_label}_{scan_index:04d}.txt"
        
        igor_path = self.windows2igor(self.file_loc)
        self.igor.Execute(f'NewPath/O TuneSavePath "{igor_path}"')
        
        # Change to the Tune folder, then save
        self.igor.Execute('SetDataFolder root:packages:MFP3D:Tune')
        self.igor.Execute(
            f'Save/T/O/P=TuneSavePath '
            f'Frequency, Phase, Amp '
            f'as "{tune_filename}"'
        )
        # Return to root
        self.igor.Execute('SetDataFolder root:')
        
        time.sleep(0.5)
        return os.path.join(self.file_loc, tune_filename)
    
    def get_tune_data(self):
        """
        Get the tune data (Frequency, Phase, Amplitude) from Igor.
        
        Returns
        -------
        dict
            Dictionary with 'frequency', 'phase', 'amplitude' arrays
        """
        try:
            self.igor.Execute('SetDataFolder root:packages:MFP3D:Tune')
            
            freq_wave = self.igor.DataFolder(r"root:packages:MFP3D:Tune").Wave("Frequency")
            phase_wave = self.igor.DataFolder(r"root:packages:MFP3D:Tune").Wave("Phase")
            amp_wave = self.igor.DataFolder(r"root:packages:MFP3D:Tune").Wave("Amp")
            
            # Get dimensions
            n_points = freq_wave.GetDimensions()[0]
            
            # Extract data
            freq = np.array([freq_wave.GetNumericWavePointValue(i) for i in range(n_points)])
            phase = np.array([phase_wave.GetNumericWavePointValue(i) for i in range(n_points)])
            amp = np.array([amp_wave.GetNumericWavePointValue(i) for i in range(n_points)])
            
            self.igor.Execute('SetDataFolder root:')
            
            return {'frequency': freq, 'phase': phase, 'amplitude': amp}
        except Exception as e:
            print(f"    WARNING: Could not get tune data: {e}")
            self.igor.Execute('SetDataFolder root:')
            return None

    def extract_resonance_from_tune(self, tune_data):
        """
        Extract resonance frequency and Q factor from tune data.
        
        Uses peak amplitude to find resonance frequency.
        Q estimated from -3dB bandwidth.
        """
        if tune_data is None:
            return np.nan, np.nan
        
        freq = tune_data['frequency']
        amp = tune_data['amplitude']
        
        # Find peak
        peak_idx = np.argmax(amp)
        f_res = freq[peak_idx]
        amp_max = amp[peak_idx]
        
        # Estimate Q from -3dB bandwidth
        try:
            amp_3db = amp_max / np.sqrt(2)
            above_3db = amp > amp_3db
            
            # Find bandwidth
            indices = np.where(above_3db)[0]
            if len(indices) > 1:
                f_low = freq[indices[0]]
                f_high = freq[indices[-1]]
                bandwidth = f_high - f_low
                Q = f_res / bandwidth if bandwidth > 0 else np.nan
            else:
                Q = np.nan
        except:
            Q = np.nan
        
        return f_res, Q

    def tune_eigenmode(self, position_label="", scan_index=0, save_tune_data=True):
        """
        Perform a tune (using manually set frequency range in Igor).
        
        Parameters
        ----------
        position_label : str, optional
            Label for saving tune data
        scan_index : int, optional
            Index for filename
        save_tune_data : bool, optional
            If True, save tune data to file
        
        Returns
        -------
        dict
            Tune results including resonance frequency, Q factor, etc.
        """
        print(f"  Performing tune...")
        
        # Perform the tune
        self.do_tune(wait_time=self.tune_settling_time + 2)
        
        # Get tune data and extract resonance
        tune_data = self.get_tune_data()
        f_res, Q = self.extract_resonance_from_tune(tune_data)
        
        tune_results = {
            'resonance_freq': f_res,
            'q_factor': Q,
            'tune_data': tune_data
        }
        
        # Save tune data
        if save_tune_data:
            try:
                tune_file = self.save_tune(position_label, scan_index)
                tune_results['tune_file'] = tune_file
                print(f"    Tune saved: {os.path.basename(tune_file)}")
            except Exception as e:
                print(f"    WARNING: Could not save tune data: {e}")
        
        if not np.isnan(f_res):
            print(f"    Resonance: {f_res/1000:.2f} kHz")
        if not np.isnan(Q):
            print(f"    Q factor: {Q:.1f}")
        
        return tune_results
   
    def run_single_position(self, position_label="", f=None):
        """
        Run the full measurement sequence at the current position.
        
        Workflow:
        1. Capture optical image
        2. Force curve for InvOLS
        3. Optional AutoWedge
        4. Engage and tune eigenmode
        """
        position_results = {
            'label': position_label,
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
        }
        
        self.saveprint(f"\n{'='*50}", f)
        self.saveprint(f"Position: {position_label}", f)
        self.saveprint(f"{'='*50}", f)
        
        # Step 1: Capture optical image
        self.saveprint("\n[Step 1] Capturing optical image", f)
        try:
            image_result = self.capture_optical_image(position_label)
            position_results['optical_image'] = image_result
        except Exception as e:
            self.saveprint(f"  ERROR capturing image: {e}", f)
            position_results['optical_image_error'] = str(e)
        
        # Step 2: Force curve for InvOLS
        self.saveprint("\n[Step 2] Measuring InvOLS (force curve)", f)
        try:
            invols = self.measure_invols()
            position_results['invols'] = invols
            if invols:
                self.saveprint(f"    InvOLS: {invols:.3e} m/V", f)
        except Exception as e:
            self.saveprint(f"  ERROR: {e}", f)
            position_results['invols'] = None
            position_results['invols_error'] = str(e)
        
        # Step 3: Optional AutoWedge
        if self.use_autowedge:
            self.saveprint(f"\n[Step 3] Running AutoWedge (with {self.autowedge_pause}s pause)", f)
            try:
                self.do_autowedge()
                position_results['autowedge'] = True
            except Exception as e:
                self.saveprint(f"  ERROR: {e}", f)
                position_results['autowedge_error'] = str(e)
        else:
            self.saveprint("\n[Step 3] AutoWedge: SKIPPED", f)
            position_results['autowedge'] = False
        
        # Step 4: Engage and Tune
        self.saveprint("\n[Step 4] Engaging and tuning eigenmode", f)
        try:
            # Engage
            self.simple_engage()
            position_results['engaged'] = True
            
            # Tune
            tune_result = self.tune_eigenmode(
                position_label=position_label,
                scan_index=getattr(self, '_current_scan_index', 0)
            )
            position_results['tune'] = tune_result
            if not np.isnan(tune_result['resonance_freq']):
                self.saveprint(f"    Resonance: {tune_result['resonance_freq']/1000:.2f} kHz", f)
            if not np.isnan(tune_result['q_factor']):
                self.saveprint(f"    Q: {tune_result['q_factor']:.1f}", f)
            
            # Withdraw after tuning
            self.withdraw()
                
        except Exception as e:
            self.saveprint(f"  ERROR: {e}", f)
            position_results['tune_error'] = str(e)
            # Try to withdraw anyway
            try:
                self.withdraw()
            except:
                pass
        
        return position_results
    
    def run_laser_sweep(self, positions, return_to_start=True):
        """
        Run the full laser sweep automation.
        
        Parameters
        ----------
        positions : list of tuples
            List of (x_um, y_um) positions to measure
        return_to_start : bool
            If True, return laser to starting position after sweep
        """
        all_results = []
        cumulative_x = 0
        cumulative_y = 0
        
        with open(self.log_filename + '.txt', 'a') as f:
            self.saveprint(f"\n{'#'*60}", f)
            self.saveprint(f"# Laser Sweep Started: {time.strftime('%Y-%m-%d %H:%M:%S')}", f)
            self.saveprint(f"# Positions: {len(positions)}", f)
            self.saveprint(f"# AutoWedge: {'ENABLED' if self.use_autowedge else 'DISABLED'}", f)
            self.saveprint(f"{'#'*60}\n", f)
            
            for i, (x_um, y_um) in enumerate(positions):
                self.saveprint(f"\n>>> Position {i+1}/{len(positions)}: ({x_um}, {y_um}) µm", f)
                
                # Move laser (relative from previous position)
                if i == 0:
                    dx, dy = x_um, y_um
                else:
                    prev_x, prev_y = positions[i-1]
                    dx = x_um - prev_x
                    dy = y_um - prev_y
                
                if dx != 0 or dy != 0:
                    self.saveprint(f"    Moving laser by ({dx}, {dy}) µm...", f)
                    self.move_laser_to_position(dx, dy, relative=True)
                    time.sleep(1)
                
                cumulative_x += dx
                cumulative_y += dy
                
                # Run measurements at this position
                position_label = f"Pos_{i+1}_x{x_um}_y{y_um}"
                self._current_scan_index = i
                results = self.run_single_position(position_label, f)
                results['position_um'] = (x_um, y_um)
                results['cumulative_position_um'] = (cumulative_x, cumulative_y)
                
                all_results.append(results)
                
                # Summary for this position
                self.saveprint(f"\n    --- Position {i+1} Summary ---", f)
                self.saveprint(f"    InvOLS: {results.get('invols', 'N/A')}", f)
                if 'tune' in results:
                    f_res = results['tune'].get('resonance_freq', np.nan)
                    q_val = results['tune'].get('q_factor', np.nan)
                    if not np.isnan(f_res):
                        self.saveprint(f"    Resonance: {f_res/1000:.2f} kHz", f)
                    if not np.isnan(q_val):
                        self.saveprint(f"    Q factor: {q_val:.1f}", f)
            
            # Return to start if requested
            if return_to_start and (cumulative_x != 0 or cumulative_y != 0):
                self.saveprint(f"\n>>> Returning to start position...", f)
                self.move_laser_to_position(-cumulative_x, -cumulative_y, relative=True)
                self.saveprint(f"    Moved by ({-cumulative_x}, {-cumulative_y}) µm", f)
            
            self.saveprint(f"\n{'#'*60}", f)
            self.saveprint(f"# Sweep Complete: {time.strftime('%Y-%m-%d %H:%M:%S')}", f)
            self.saveprint(f"# Total positions: {len(all_results)}", f)
            self.saveprint(f"{'#'*60}\n", f)
        
        self.results = all_results
        return all_results

In [43]:
#to do a scan:
def do_scan(igor, wait=False, mode = None, file_loc = None, base_filename = None):
    if file_loc:
        set_folder(igor, file_loc)
    else:
        file_loc = check_folder(igor)
    if base_filename:
        set_base_filename(igor, base_filename)
    else:
        base_filename = check_base_filename(igor)
    if mode:
        ex(igor, 'LastScanPopup_0', "MasterPanel", 0, mode)
    filename = get_next_filename(base_filename, file_loc, '.ibw')
    
    ex(igor, 'DownScan_0','MasterPanel')
    time.sleep(10)
    if wait:
        while scanning(igor):
            time.sleep(1)
    if file_loc[-1] != '\\':
        file_loc+= '\\'
    full_path = file_loc+filename
    return full_path

#to chack if its scanning 
def scanning(igor):
    data = igor.DataFolder(r"root:packages:MFP3D").Wave("OutWaves")
    outputting = bool(data.GetTextWavePointValue(0,0))
    return outputting

def check_folder(igor):
    file_loc_igor = igor.DataFolder(r"root:Packages:MFP3D:Main:Strings").Wave("GlobalStrings").GetTextWavePointValue(18, 0)
    return igor2windows(file_loc_igor)

def igor2windows(file_loc_igor):
    file_loc = file_loc_igor.split(':', 1)
    file_loc[1] = file_loc[1].replace(':', '\\')
    file_loc = file_loc[0]+':\\'+file_loc[1]
    return file_loc

def check_base_filename(igor):
    return igor.DataFolder(r"root:packages:MFP3D:Main:Variables").Variable("BaseName").GetStringValue(0)


def get_next_filename(base_name, directory, filetype):
    """
    Generates the next available filename with a zero-padded numeric suffix,
    based on all files starting with the base name, regardless of extension.
    
    Parameters:
        base_name (str): The base filename (e.g., "Test").
        directory (str): The directory to search in.
        filetype (str): The file extension to return (e.g., ".ibw").
    
    Returns:
        str: The next available filename (e.g., "Test0002.ibw").
    """
    pattern = re.compile(rf"^{re.escape(base_name)}(\d{{4}})\..+$")
    max_index = -1

    for filename in os.listdir(directory):
        match = pattern.match(filename)
        if match:
            index = int(match.group(1))
            if index > max_index:
                max_index = index

    next_index = max_index + 1
    return f"{base_name}{next_index:04d}{filetype}"

def ex(igor, variable = "", panel = "", val = 0, string = "", verbose=False):
    execution_line = ""
    if verbose:
        execution_line += "print "
    execution_line += 'ARExecuteControl("'
    execution_line += variable
    execution_line += '", "'
    execution_line += panel
    execution_line += '", '
    execution_line += str(val)
    execution_line += ', "'
    execution_line += string
    execution_line += '")'
    igor.Execute(execution_line)
    return execution_line

## 5. Initialize Automation

In [44]:
# Create automation instance
automation = AFMLaserSweepAutomation(igor, file_loc, base_filename, log_filename)

# Configure parameters
automation.load_force_setpoint = load_setpoint
automation.eigenmode_center_freq = eigenmode_center
automation.tune_settling_time = tune_settling_time
automation.invols_bounds = (invols_min, invols_max)

# AutoWedge settings
automation.use_autowedge = use_autowedge
automation.autowedge_pause = autowedge_pause

print("Automation initialized!")
print(f"  AutoWedge: {'ENABLED' if automation.use_autowedge else 'DISABLED'}")

Automation initialized!
  AutoWedge: ENABLED


## 6. Define Positions

In [25]:
#automation.do_autowedge()
#automation.do_ld_move(30,0)

In [45]:
# Generate positions for linear X sweep
positions = [(i * step_um, 0) for i in range(num_steps)]

print("Positions to measure:")
for i, (x, y) in enumerate(positions):
    print(f"  {i+1}. ({x}, {y}) µm")

Positions to measure:
  1. (0.0, 0) µm
  2. (0.5, 0) µm
  3. (1.0, 0) µm
  4. (1.5, 0) µm
  5. (2.0, 0) µm
  6. (2.5, 0) µm
  7. (3.0, 0) µm
  8. (3.5, 0) µm
  9. (4.0, 0) µm
  10. (4.5, 0) µm
  11. (5.0, 0) µm
  12. (5.5, 0) µm
  13. (6.0, 0) µm
  14. (6.5, 0) µm
  15. (7.0, 0) µm
  16. (7.5, 0) µm
  17. (8.0, 0) µm
  18. (8.5, 0) µm
  19. (9.0, 0) µm
  20. (9.5, 0) µm
  21. (10.0, 0) µm
  22. (10.5, 0) µm
  23. (11.0, 0) µm
  24. (11.5, 0) µm
  25. (12.0, 0) µm
  26. (12.5, 0) µm
  27. (13.0, 0) µm
  28. (13.5, 0) µm
  29. (14.0, 0) µm
  30. (14.5, 0) µm
  31. (15.0, 0) µm
  32. (15.5, 0) µm
  33. (16.0, 0) µm
  34. (16.5, 0) µm
  35. (17.0, 0) µm
  36. (17.5, 0) µm
  37. (18.0, 0) µm
  38. (18.5, 0) µm
  39. (19.0, 0) µm
  40. (19.5, 0) µm
  41. (20.0, 0) µm
  42. (20.5, 0) µm
  43. (21.0, 0) µm
  44. (21.5, 0) µm
  45. (22.0, 0) µm
  46. (22.5, 0) µm
  47. (23.0, 0) µm
  48. (23.5, 0) µm
  49. (24.0, 0) µm
  50. (24.5, 0) µm
  51. (25.0, 0) µm
  52. (25.5, 0) µm
  53. (26.0, 0) µm


In [46]:
# =============================================================================
# LOAD-DEPENDENT LASER POSITION SWEEP
# =============================================================================
# Add this cell to your laser sweep notebook to perform sweeps at different loads

def run_laser_sweep_vs_load(automation, loads_nN, step_um, num_steps, 
                              eigenmode_center, log_filename, 
                              invols_from_first=True):
    """
    Perform laser position sweep at multiple loads (setpoints).
    
    Parameters
    ----------
    automation : AFMLaserSweepAutomation
        The automation object (already initialized)
    loads_nN : list of float
        List of loads in nN to test at each position
    step_um : float
        Laser step size in micrometers
    num_steps : int
        Number of laser positions to test
    eigenmode_center : float
        Center frequency for tuning (Hz)
    log_filename : str
        Base path for log file (no extension)
    invols_from_first : bool, optional
        If True, measure InvOLS only at first load and reuse for others
    
    Returns
    -------
    dict
        Results organized by load, then position
    """
    import pandas as pd
    import time
    import os
    
    # Get calibration from Igor
    gmv = automation.get_gmv()
    spring_const = gmv['SpringConstant']
    invols = gmv['InvOLS']
    
    print("\n" + "="*70)
    print("LOAD-DEPENDENT LASER POSITION SWEEP")
    print("="*70)
    print(f"Spring constant: {spring_const:.3f} N/m")
    print(f"InvOLS: {invols:.3e} m/V")
    print(f"Loads: {loads_nN} nN")
    print(f"Laser positions: {num_steps} steps of {step_um} µm")
    print("="*70)
    
    # Convert loads to setpoint voltages
    setpoints_V = []
    for load_nN in loads_nN:
        load_N = load_nN * 1e-9
        defl_m = load_N / spring_const
        setpoint_V = defl_m / invols
        setpoints_V.append(setpoint_V)
        print(f"  {load_nN:6.0f} nN → {setpoint_V:.4f} V")
    
    # Storage for all results
    all_results = {load: [] for load in loads_nN}
    
    # NOTE: We do NOT engage initially here
    # Instead, we'll engage at each load during the measurement loop
    # This is because setpoint can only be set during engagement with SimpleEngageMe
    
    # Loop over laser positions
    for pos_idx in range(num_steps):
        position_x = pos_idx * step_um
        position_y = 0
        position_label = f"X{position_x:03.0f}"
        
        print(f"\n{'='*70}")
        print(f"LASER POSITION {pos_idx+1}/{num_steps}: ({position_x:.1f}, {position_y:.1f}) µm")
        print(f"{'='*70}")
        
        # CRITICAL: Must withdraw before moving laser (motors locked when engaged)
        if pos_idx > 0:
            print(f"\nWithdrawing to move laser...")
            automation.withdraw()
            time.sleep(2)
            
            print(f"Moving laser by {step_um} µm...")
            automation.move_laser_to_position(step_um, 0, relative=True)
            time.sleep(1.5)  # Extra time for laser to settle
            print(f"  ✓ Laser moved to position ({position_x:.1f}, {position_y:.1f}) µm")
        else:
            print(f"\nStarting at initial laser position")
            # Still need to be withdrawn for first position (for InvOLS measurement)
            print("Ensuring tip is withdrawn...")
            automation.withdraw()
            time.sleep(2)
        
        # Capture optical image of cantilever at this positionwedge
        print("\n[Optical Image] Capturing cantilever image")
        try:
            automation.capture_optical_image(position_label=position_label)
            print(f"  ✓ Image captured for position {position_label}")
            print(f"  → Check image to verify laser spot position!")
        except Exception as e:
            print(f"  ⚠ Warning: Could not capture image: {e}")
        
        # Measure InvOLS at this position (only at first position or if requested)
        # We're already withdrawn from laser movement above
        if pos_idx == 0 or not invols_from_first:
            automation.do_autowedge()
            time.sleep(15)
            print("\n[Step 1] Measuring InvOLS")
            invols_measured = automation.measure_invols()
           
            # Don't re-engage here - will engage at first load below
        else:
            invols_measured = invols  # Reuse from first position
            print(f"\n[Step 1] Using InvOLS from first position: {invols_measured:.3e} m/V")
        
        # Loop over loads at this laser position
        for load_idx, (load_nN, setpoint_V) in enumerate(zip(loads_nN, setpoints_V)):
            print(f"\n  {'─'*60}")
            print(f"  LOAD {load_idx+1}/{len(loads_nN)}: {load_nN:.0f} nN ({setpoint_V:.4f} V)")
            print(f"  {'─'*60}")
            
            # CRITICAL: Must withdraw and re-engage to change load
            # Setpoint changes don't work while engaged - only during engagement
            
            if load_idx > 0:
                print(f"  Withdrawing to change load...")
                automation.withdraw()
                time.sleep(2)
            
            # Set the new setpoint
            print(f"  Setting deflection setpoint: {setpoint_V:.4f} V")
            automation.igor.Execute(f'PV("DeflectionSetpointVolts", {setpoint_V})')
            time.sleep(0.5)
            
            # Engage with the new setpoint
            print(f"  Engaging at {load_nN:.0f} nN...")
            automation.simple_engage(wait_time=5)
            time.sleep(2)
            
            # Verify engagement
            gmv = automation.get_gmv()
            current_setpoint = gmv['DeflectionSetpointVolts']
            print(f"  ✓ Engaged at setpoint: {current_setpoint:.4f} V")
            
            # Perform tune
            print(f"\n  [Step 2] Tuning eigenmode at {load_nN:.0f} nN")
            scan_index = pos_idx * 1000 + load_idx
            tune_results = automation.tune_eigenmode(
                position_label=f"{position_label}_L{load_nN:04.0f}nN",
                scan_index=scan_index,
                save_tune_data=True
            )
            
            # Store results
            result_entry = {
                'position_x_um': position_x,
                'position_y_um': position_y,
                'load_nN': load_nN,
                'setpoint_V': setpoint_V,
                'invols_m_per_V': invols_measured,
                'resonance_freq_Hz': tune_results['resonance_freq'],
                'q_factor': tune_results['q_factor'],
                'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
            }
            all_results[load_nN].append(result_entry)
            
            if not np.isnan(tune_results['resonance_freq']):
                print(f"  ✓ f_res = {tune_results['resonance_freq']/1000:.2f} kHz")
            if not np.isnan(tune_results['q_factor']):
                print(f"  ✓ Q = {tune_results['q_factor']:.1f}")
    
    # Withdraw at end
    print(f"\n{'='*70}")
    print("EXPERIMENT COMPLETE - Withdrawing")
    print(f"{'='*70}")
    automation.withdraw()
    
    # Save results to CSV files (one per load)
    print(f"\n{'='*70}")
    print("SAVING RESULTS")
    print(f"{'='*70}")
    
    base_dir = os.path.dirname(log_filename)
    base_name = os.path.basename(log_filename)
    
    for load_nN in loads_nN:
        df = pd.DataFrame(all_results[load_nN])
        csv_path = os.path.join(base_dir, f"{base_name}_Load{load_nN:04.0f}nN.csv")
        df.to_csv(csv_path, index=False)
        print(f"✓ Saved: {os.path.basename(csv_path)}")
    
    # Also save combined file
    combined_results = []
    for load_nN in loads_nN:
        combined_results.extend(all_results[load_nN])
    
    df_combined = pd.DataFrame(combined_results)
    csv_combined = os.path.join(base_dir, f"{base_name}_AllLoads.csv")
    df_combined.to_csv(csv_combined, index=False)
    print(f"✓ Saved: {os.path.basename(csv_combined)}")
    
    return all_results


# =============================================================================
# EXAMPLE USAGE
# =============================================================================
"""
# Define loads to test (in nN)
loads_nN = [100, 200, 500, 1000, 2000]

# Run the load-dependent sweep
results = run_laser_sweep_vs_load(
    automation=automation,
    loads_nN=loads_nN,
    step_um=step_um,
    num_steps=num_steps,
    eigenmode_center=eigenmode_center,
    log_filename=log_filename,
    invols_from_first=True  # Measure InvOLS only at first position
)

# Results are saved as CSV files:
# - LaserSweep_log_Load0100nN.csv
# - LaserSweep_log_Load0200nN.csv
# - ...
# - LaserSweep_log_AllLoads.csv (combined)
"""


# =============================================================================
# PLOTTING FUNCTION
# =============================================================================
def plot_laser_sweep_vs_load(results, loads_nN, save_path=None):
    """
    Plot resonance frequency vs laser position for multiple loads.
    
    Parameters
    ----------
    results : dict
        Results dictionary from run_laser_sweep_vs_load()
    loads_nN : list
        List of loads tested
    save_path : str, optional
        Path to save figure
    """
    import matplotlib.pyplot as plt
    import numpy as np
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(loads_nN)))
    
    # Plot resonance frequency
    ax = axes[0]
    for i, load_nN in enumerate(loads_nN):
        data = results[load_nN]
        positions = [r['position_x_um'] for r in data]
        freqs = [r['resonance_freq_Hz']/1000 for r in data]
        
        ax.plot(positions, freqs, 'o-', color=colors[i], 
                label=f'{load_nN:.0f} nN', linewidth=2, markersize=6)
    
    ax.set_xlabel('Laser Position (µm)', fontsize=11)
    ax.set_ylabel('Resonance Frequency (kHz)', fontsize=11)
    ax.set_title('Contact Resonance vs Laser Position', fontweight='bold')
    ax.legend(title='Load', fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Plot Q factor
    ax = axes[1]
    for i, load_nN in enumerate(loads_nN):
        data = results[load_nN]
        positions = [r['position_x_um'] for r in data]
        qs = [r['q_factor'] for r in data]
        
        ax.plot(positions, qs, 'o-', color=colors[i], 
                label=f'{load_nN:.0f} nN', linewidth=2, markersize=6)
    
    ax.set_xlabel('Laser Position (µm)', fontsize=11)
    ax.set_ylabel('Q Factor', fontsize=11)
    ax.set_title('Q Factor vs Laser Position', fontweight='bold')
    ax.legend(title='Load', fontsize=9)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure saved: {save_path}")
    
    plt.show()


"""
# EXAMPLE: Plot results
plot_laser_sweep_vs_load(
    results=results,
    loads_nN=loads_nN,
    save_path=os.path.join(file_loc, f'{base_filename}_vs_Load.png')
)
"""

"\n# EXAMPLE: Plot results\nplot_laser_sweep_vs_load(\n    results=results,\n    loads_nN=loads_nN,\n    save_path=os.path.join(file_loc, f'{base_filename}_vs_Load.png')\n)\n"

In [47]:
def run_laser_sweep_single_load_bias(automation, load_nN, dc_bias_V, step_um, num_steps,
                                        eigenmode_center, log_filename,
                                        image_freq_Hz=35000,
                                        resonance_freq_Hz=400000):
    """
    Perform laser position sweep at a single DC bias and single load.

    At each laser position the workflow is:
        1. Capture optical image (video panel)
        2. AutoWedge + InvOLS measurement
        3. Engage at the requested load
        4. Tune eigenmode (saves tune file)
        5. Acquire one AFM scan at image_freq_Hz (e.g. 35 kHz)
        6. Reset drive frequency to resonance_freq_Hz (e.g. 400 kHz)
        7. Withdraw and move to next position

    Parameters
    ----------
    automation : AFMLaserSweepAutomation
    load_nN : float
        Load in nN.
    dc_bias_V : float
        DC bias voltage for Output.A (V).
    step_um : float
        Laser step size (µm).
    num_steps : int
        Number of laser positions.
    eigenmode_center : float
        Tune centre frequency (Hz).
    log_filename : str
        Base path for log file (no extension).
    image_freq_Hz : float, optional
        Drive frequency (Hz) used while acquiring the AFM image.
        Default: 35 000 Hz.
    resonance_freq_Hz : float, optional
        Drive frequency (Hz) to restore after the image.
        Default: 400 000 Hz.

    Returns
    -------
    list of dict
        One result dict per laser position.
    """
    import numpy as np
    import time

    # ------------------------------------------------------------------ #
    #  Initial calibration                                                 #
    # ------------------------------------------------------------------ #
    gmv          = automation.get_gmv()
    spring_const = gmv['SpringConstant']
    invols_init  = gmv['InvOLS']

    # Convert load to setpoint voltage
    setpoint_V = (load_nN * 1e-9) / spring_const / invols_init

    print("\n" + "=" * 70)
    print("LASER SWEEP  ·  SINGLE LOAD  ·  SINGLE DC BIAS")
    print("=" * 70)
    print(f"Spring constant : {spring_const:.3f} N/m")
    print(f"InvOLS (init)   : {invols_init:.3e} m/V")
    print(f"Load            : {load_nN} nN  →  {setpoint_V:.4f} V")
    print(f"DC bias         : {dc_bias_V:+.3f} V")
    print(f"Laser positions : {num_steps} steps of {step_um} µm")
    print(f"Image frequency : {image_freq_Hz/1e3:.1f} kHz")
    print(f"Resonance (tune): {resonance_freq_Hz/1e3:.1f} kHz")
    print("=" * 70)

    all_results = []

    # Apply DC bias once (stays constant throughout)
    print(f"\nSetting Output.A → {dc_bias_V:+.3f} V")
    automation.igor.Execute(f'td_wv("Cypher.Output.A", {dc_bias_V})')
    time.sleep(1.0)

    # ================================================================== #
    #  MAIN LOOP: laser positions                                          #
    # ================================================================== #
    for pos_idx in range(num_steps):
        position_x     = pos_idx * step_um
        position_label = f"X{1e3 * position_x:03.0f}"

        print(f"\n{'#' * 70}")
        print(f"  LASER POSITION {pos_idx + 1}/{num_steps}: x = {position_x:.1f} µm")
        print(f"{'#' * 70}")

        # --- withdraw before touching the laser motor ---
        automation.withdraw()
        time.sleep(2)

        if pos_idx == 0:
            print("  Starting at initial laser position (tip withdrawn).")
        else:
            print(f"  Moving laser by {step_um} µm...")
            automation.move_laser_to_position(step_um, 0, relative=True)
            time.sleep(1.5)
            print(f"  ✓ Laser at x = {position_x:.1f} µm")

        # ---- Step 1: optical image ----
        print("\n  [Step 1] Capturing optical image")
        try:
            automation.capture_optical_image(position_label=position_label)
            print("  ✓ Image captured — verify laser spot position!")
        except Exception as e:
            print(f"  ⚠ Warning: could not capture image: {e}")
        igor.Execute("DoForceFunc(\"GoForce_1\")") 
        print("  Moving 2 center")
        # ---- Step 2: AutoWedge + InvOLS ----
        automation.do_autowedge()
        time.sleep(10)
        print("\n  [Step 2] Measuring InvOLS")
        invols_measured = automation.measure_invols()
        print(f"  ✓ InvOLS = {invols_measured:.3e} m/V")

        # Recompute setpoint with fresh InvOLS
        sp = (load_nN * 1e-9) / spring_const / invols_measured

        # ---- Step 3: engage ----
        print(f"\n  [Step 3] Setting setpoint = {sp:.4f} V and engaging at {load_nN:.0f} nN")
        automation.igor.Execute
        
          
        automation.igor.Execute(f'PV("DeflectionSetpointVolts", {sp})')
        time.sleep(0.5)
        automation.simple_engage(wait_time=5)
        time.sleep(2)

        gmv_now = automation.get_gmv()
        print(f"  ✓ Engaged — setpoint reads {gmv_now['DeflectionSetpointVolts']:.4f} V")

        # ---- Step 4: tune eigenmode ----
        bias_str   = f"{'m' if dc_bias_V < 0 else 'p'}{abs(dc_bias_V):.3f}V".replace(".", "p")
        scan_index = pos_idx

        print("\n  [Step 4] Tuning eigenmode")
        tune_results = automation.tune_eigenmode(
            position_label=f"{position_label}_DC{bias_str}_L{load_nN:04.0f}nN",
            scan_index=scan_index,
            save_tune_data=True,
        )

        f_res = tune_results['resonance_freq']
        q     = tune_results['q_factor']
        if not np.isnan(f_res):
            print(f"  ✓ f_res = {f_res / 1e3:.2f} kHz")
        if not np.isnan(q):
            print(f"  ✓ Q     = {q:.1f}")

        # ---- Step 5: acquire AFM image at 35 kHz ----
        print(f"\n  [Step 5] Acquiring AFM image at {image_freq_Hz/1e3:.1f} kHz")
        automation.igor.Execute(f'PV("DriveFrequency", {image_freq_Hz})')
        time.sleep(0.5)
        try:
            image_path = do_scan(igor=automation.igor, wait=True)
            print(f"  ✓ Image saved: {image_path}")
        except Exception as e:
            print(f"  ⚠ Warning: image acquisition failed: {e}")
            image_path = None

        # ---- Step 6: restore drive frequency to resonance ----
        print(f"  [Step 6] Restoring drive frequency to {resonance_freq_Hz/1e3:.1f} kHz")
        automation.igor.Execute(f'PV("DriveFrequency", {resonance_freq_Hz})')
        time.sleep(0.5)
        print(f"  ✓ Drive frequency reset to {resonance_freq_Hz/1e3:.1f} kHz")

        # ---- Withdraw ----
        automation.withdraw()
        time.sleep(1)

        # ---- Store result ----
        result_entry = {
            'position_x_um'    : position_x,
            'dc_bias_V'        : dc_bias_V,
            'load_nN'          : load_nN,
            'setpoint_V'       : sp,
            'invols_m_per_V'   : invols_measured,
            'resonance_freq_Hz': f_res,
            'q_factor'         : q,
            'image_path'       : image_path,
            'timestamp'        : time.strftime('%Y-%m-%d %H:%M:%S'),
        }
        all_results.append(result_entry)

    # ================================================================== #
    #  Done                                                                #
    # ================================================================== #
    print(f"\n{'#' * 70}")
    print("ALL POSITIONS COMPLETE — withdrawing and zeroing DC bias")
    print(f"{'#' * 70}")

    automation.withdraw()
    automation.igor.Execute('td_wv("Cypher.Output.A", 0)')
    print("✓ Output.A reset to 0 V")

    # Save results to CSV
    import pandas as pd, os
    df = pd.DataFrame(all_results)
    base_dir  = os.path.dirname(log_filename)
    base_name = os.path.basename(log_filename)
    bias_str_file = f"{'m' if dc_bias_V < 0 else 'p'}{abs(dc_bias_V):.3f}V".replace(".", "p")
    csv_path = os.path.join(base_dir,
                            f"{base_name}_DC{bias_str_file}_L{load_nN:04.0f}nN.csv")
    df.to_csv(csv_path, index=False)
    print(f"✓ Results saved: {os.path.basename(csv_path)}")

    return all_results


## 7. Run the Laser Sweep

In [48]:
igor.Execute("ZeroPD(\"\")")


igor.Execute("DoForceFunc(\"GoForce_1\")")    
        

In [30]:
# Run the full laser sweep for a fixed setpoint
#results = automation.run_laser_sweep(
#    positions,
#    return_to_start=True
#)
igor.Execute("ZeroPD(\"\")")
#print(f"\nSweep complete! {len(results)} positions measured.")

In [31]:
igor.Execute("print td_wv(\"Cypher.Output.A\",0)")

In [32]:
load_nN    = 500       # single load in nN
dc_bias_V  = 2        # single DC bias in V

results = run_laser_sweep_single_load_bias(
    automation=automation,
    load_nN=load_nN,
    dc_bias_V=dc_bias_V,
    step_um=1,
    num_steps=30,
    eigenmode_center=382028,
    log_filename=log_filename,
    image_freq_Hz=28000,       # frequency for AFM image acquisition
    resonance_freq_Hz=384677,  # frequency to restore tunes after image
)



LASER SWEEP  ·  SINGLE LOAD  ·  SINGLE DC BIAS
Spring constant : 1.425 N/m
InvOLS (init)   : 4.711e-07 m/V
Load            : 500 nN  →  0.7449 V
DC bias         : +2.000 V
Laser positions : 30 steps of 1 µm
Image frequency : 28.0 kHz
Resonance (tune): 384.7 kHz

Setting Output.A → +2.000 V

######################################################################
  LASER POSITION 1/30: x = 0.0 µm
######################################################################
  Withdrawing tip...
  Starting at initial laser position (tip withdrawn).

  [Step 1] Capturing optical image
  Captured optical image
  ✓ Image captured — verify laser spot position!
  Moving 2 center
  Running AutoWedge...
  Waiting 15s for AutoWedge to settle...
  AutoWedge complete

  [Step 2] Measuring InvOLS
  Performing force curve: FLaserSweep0080.ibw
  Measured InvOLS: 5.751e-07 m/V
  InvOLS set successfully
  ✓ InvOLS = 5.751e-07 m/V

  [Step 3] Setting setpoint = 0.6101 V and engaging at 500 nN
  Engaging with Sim

In [33]:
automation.do_ld_move(-30, 0)

In [ ]:
# ==============================================================================
# MULTI-LOAD SEQUENTIAL SWEEP
# ==============================================================================
# Runs run_laser_sweep_single_load_bias repeatedly for each load in loads_nN.
# Between each run:
#   1. Repositions laser back 30 um (DoLDMove(-30, 0))
#   2. Creates a new save folder named after the load
#   3. Resets the Igor base filename and suffix to 0
# ==============================================================================

loads_nN = [250, 500, 750, 1000, 1500, 2000, 2500]     # <-- set your load sequence here

dc_bias_V         = 0
step_um           = 1
num_steps         = 40
eigenmode_center  = 317355
image_freq_Hz     = 28000
resonance_freq_Hz = 317355 

# Base directory — subfolders will be created here for each load
base_dir = r'D:\User Data\MARTI\2026\April\16th\Adama vs Load'

all_load_results = {}

for i, load_nN in enumerate(loads_nN):

    print(f"\n{'#'*70}")
    print(f"  STARTING SWEEP  {i+1}/{len(loads_nN)}:  load = {load_nN:.0f} nN")
    print(f"{'#'*70}")

    # ------------------------------------------------------------------ #
    # 1. Reposition laser back 30 um (skip before first run)
    # ------------------------------------------------------------------ #
    if i > 0:
        print(f"\n  Repositioning laser -30 um back to start...")
        automation.do_ld_move(-40, 0)
        time.sleep(2)
        print(f"  ✓ Laser repositioned")

    # ------------------------------------------------------------------ #
    # 2. Create new save folder for this load
    # ------------------------------------------------------------------ #
    load_folder = os.path.join(base_dir, f'{load_nN:.0f}nN_0Deg')
    os.makedirs(load_folder, exist_ok=True)
    print(f"\n  Save folder: {load_folder}")

    # Update automation object to point at new folder
    automation.file_loc   = load_folder
    automation.set_folder()

    # ------------------------------------------------------------------ #
    # 3. Reset Igor base filename and file suffix to 0
    # ------------------------------------------------------------------ #
    automation.igor.Execute(f'root:Packages:MFP3D:Main:Variables:BaseName = "LaserSweep"')
    automation.igor.Execute('PV("BaseSuffix", 0000)')
    automation.igor.Execute('ARCheckSuffix()')
    print(f"  ✓ Filename reset to LaserSweep0000")

    # ------------------------------------------------------------------ #
    # 4. Run the sweep
    # ------------------------------------------------------------------ #
    log_filename = os.path.join(load_folder, f'LaserSweep_log')

    results = run_laser_sweep_single_load_bias(
        automation=automation,
        load_nN=load_nN,
        dc_bias_V=dc_bias_V,
        step_um=step_um,
        num_steps=num_steps,
        eigenmode_center=eigenmode_center,
        log_filename=log_filename,
        image_freq_Hz=image_freq_Hz,
        resonance_freq_Hz=resonance_freq_Hz,
    )

    all_load_results[load_nN] = results
    print(f"\n  ✓ Sweep complete for {load_nN:.0f} nN — {len(results)} positions saved to {load_folder}")

print(f"\n{'#'*70}")
print(f"  ALL LOADS COMPLETE")
for load_nN, res in all_load_results.items():
    print(f"    {load_nN:.0f} nN  →  {len(res)} positions")
print(f"{'#'*70}")


######################################################################
  STARTING SWEEP  1/7:  load = 250 nN
######################################################################

  Save folder: D:\User Data\MARTI\2026\April\16th\Adama vs Load\250nN_0Deg
  ✓ Filename reset to LaserSweep0000

LASER SWEEP  ·  SINGLE LOAD  ·  SINGLE DC BIAS
Spring constant : 1.506 N/m
InvOLS (init)   : 4.585e-07 m/V
Load            : 250 nN  →  0.3622 V
DC bias         : +0.000 V
Laser positions : 40 steps of 1 µm
Image frequency : 28.0 kHz
Resonance (tune): 317.4 kHz

Setting Output.A → +0.000 V

######################################################################
  LASER POSITION 1/40: x = 0.0 µm
######################################################################
  Withdrawing tip...
  Starting at initial laser position (tip withdrawn).

  [Step 1] Capturing optical image
  Captured optical image
  ✓ Image captured — verify laser spot position!
  Moving 2 center
  Running AutoWedge...
  Waiting 1